# 🧪 Plugins Test — MrCrayfish Vehicle Plugin pe Colab

Server **Paper 1.21.4** cu **MrCrayfishVehiclePlugin** + **panou web de control**, rulat din acest notebook și publicat prin tunelul **FRP** (serverul tău `frps`): portul de joc `25565` + panoul web pe `8081`.

> ⚠️ **Citește o dată**
> 1. **Colab interzice prin ToS servere de jocuri** → folosește un **cont Google secundar**.
> 2. Runtime gratuit: max **~12h**, deconectare la **~90 min inactivitate**, fila trebuie să rămână deschisă. Progresul se păstrează în Google Drive (celula de oprire).
> 3. Latență ~120–180 ms din Europa (mașinile Colab sunt în SUA).
> 4. **Niciun secret nu stă în notebook.** Înainte de prima rulare creează în Colab → iconița **🔑 Secrets** intrările: `FRPS_ADDR`, `FRPS_PORT` (implicit `7000`), `FRPS_TOKEN` — exact valorile de pe serverul tău `frps`.
> 5. Tunelul e public: whitelist + `online-mode` rămân activate mereu.

**Ordinea:** *Runtime → Run all*. Prima pornire: ~4–6 minute.

In [13]:
#@title ⚙️ Configurare — completează și rulează { display-mode: "form" }
import os

#@markdown ### Minecraft / plugin
MC_VERSION = "1.21.4" #@param {type:"string"}
RELEASE_TAG = "vehicle-plugin-prototype-v0.1.0" #@param {type:"string"}
REPO_BRANCH = "arena/2f9f7b9e-mrcrayfishvehiclemod" #@param {type:"string"}
RAM = "6G" #@param ["4G", "6G", "8G", "10G"]

#@markdown ### Tunel FRP (gol = se citește din Colab Secrets)
FRPS_ADDR = "" #@param {type:"string"}
FRPS_PORT = 0 #@param {type:"integer"}
FRPS_TOKEN = "" #@param {type:"string"}
FRPS_GAME_PORT = 25565 #@param {type:"integer"}
FRPS_PANEL_PORT = 8081 #@param {type:"integer"}

#@markdown ### Jucători (OP + whitelist), separate prin virgulă
PLAYER_NAMES = "ExMarius" #@param {type:"string"}

#@markdown ### Persistență în Google Drive
PERSIST_TO_DRIVE = True #@param {type:"boolean"}

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

FRPS_ADDR = FRPS_ADDR or _secret("FRPS_ADDR")
FRPS_PORT = FRPS_PORT or int(_secret("FRPS_PORT") or 7000)
FRPS_TOKEN = FRPS_TOKEN or _secret("FRPS_TOKEN")

ROOT = "/content/mcserver"
os.makedirs(ROOT, exist_ok=True)
ops = " ".join(n.strip() for n in PLAYER_NAMES.split(",") if n.strip())

with open(f"{ROOT}/env.sh", "w") as f:
    for k, v in [("MC_VERSION", MC_VERSION), ("RELEASE_TAG", RELEASE_TAG),
                 ("REPO_BRANCH", REPO_BRANCH), ("RAM", RAM),
                 ("FRPS_ADDR", FRPS_ADDR), ("FRPS_PORT", str(FRPS_PORT)),
                 ("FRPS_TOKEN", FRPS_TOKEN), ("FRPS_GAME_PORT", str(FRPS_GAME_PORT)),
                 ("FRPS_PANEL_PORT", str(FRPS_PANEL_PORT)), ("OP_LIST", ops),
                 ("ROOT", ROOT)]:
        f.write(f'{k}="{v}"\n')

assert FRPS_ADDR, "❌ FRPS_ADDR gol: pune-l în Colab Secrets sau în câmpul de mai sus."
assert FRPS_TOKEN, "❌ FRPS_TOKEN gol: pune-l în Colab Secrets sau în câmpul de mai sus."
print("✅ Configurație OK")
print(f"   Joc:   {FRPS_ADDR}:{FRPS_GAME_PORT}")
print(f"   Panou: http://{FRPS_ADDR}:{FRPS_PANEL_PORT}")
print(f"   OP:    {ops or '(completează PLAYER_NAMES!)'}   RAM: {RAM}")


✅ Configurație OK
   Joc:   140.238.218.210:25565
   Panou: http://140.238.218.210:8081
   OP:    ExMarius   RAM: 6G


## 1. Dependințe
Java 21 + `screen`. Atât — panoul e scris doar cu biblioteca standard Python (fără Flask/pip).

In [14]:
%%bash
set -e
source /content/mcserver/env.sh
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jre-headless screen curl git >/dev/null
update-alternatives --install /usr/bin/java java /usr/lib/jvm/java-21-openjdk-amd64/bin/java 2100 >/dev/null
update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java >/dev/null
java -version 2>&1 | head -1
echo "✅ screen: $(screen --version | head -1)"


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
✅ screen: Screen version 4.09.01 (GNU) 20-Aug-23


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Panoul web + scripturile repo-ului
Se clonează branch-ul curent și se instalează `oracle-panel/` în `/content/mcserver` (`bin/`, `panel/`), plus `frpc` v0.60.0 — versiunea probată anterior cu acest `frps`. RAM-ul ales se aplică în copia locală a scriptului de pornire.

In [15]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
rm -rf /content/repo
git clone --depth 1 --branch "$REPO_BRANCH" --quiet https://github.com/ExMarius/MrCrayfishVehicleMod.git /content/repo

cd "$ROOT"
mkdir -p bin panel logs mc frp
cp /content/repo/oracle-panel/scripts/*.sh bin/
cp /content/repo/oracle-panel/panel.py panel/
chmod +x bin/*.sh
sed -i "s/-Xmx2G/-Xmx${RAM}/" bin/start-minecraft.sh
echo "✅ oracle-panel instalat; start-minecraft.sh folosește -Xmx${RAM}"

F=frp_0.60.0_linux_amd64
[ -x frp/frpc ] || { curl -fsSL -o /tmp/frp.tgz "https://github.com/fatedier/frp/releases/download/v0.60.0/${F}.tar.gz"
  tar -xzf /tmp/frp.tgz -C /tmp && cp "/tmp/${F}/frpc" frp/frpc && chmod +x frp/frpc; }
frp/frpc --version


✅ oracle-panel instalat; start-minecraft.sh folosește -Xmx6G
0.60.0


## 3. Paper
Ultimul build pentru versiunea aleasă, verificat SHA-256, salvat ca `mc/server.jar` (locația așteptată de scripturile repo-ului).

In [16]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
cd "$ROOT/mc"
echo "→ Caut ultimul build Paper $MC_VERSION…"
B=$(curl -fsSL "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds")
BUILD=$(echo "$B" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["build"])')
JAR=$(echo "$B" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["name"])')
SHA=$(echo "$B" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["sha256"])')
[ -f server.jar ] || curl -fsSL -o server.jar "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds/$BUILD/downloads/$JAR"
echo "$SHA  server.jar" | sha256sum -c -
ls -la server.jar


→ Caut ultimul build Paper 1.21.4…


curl: (22) The requested URL returned error: 410


CalledProcessError: Command 'b'set -euo pipefail\nsource /content/mcserver/env.sh\ncd "$ROOT/mc"\necho "\xe2\x86\x92 Caut ultimul build Paper $MC_VERSION\xe2\x80\xa6"\nB=$(curl -fsSL "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds")\nBUILD=$(echo "$B" | python3 -c \'import sys,json; print(json.load(sys.stdin)["builds"][-1]["build"])\')\nJAR=$(echo "$B" | python3 -c \'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["name"])\')\nSHA=$(echo "$B" | python3 -c \'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["sha256"])\')\n[ -f server.jar ] || curl -fsSL -o server.jar "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds/$BUILD/downloads/$JAR"\necho "$SHA  server.jar" | sha256sum -c -\nls -la server.jar\n'' returned non-zero exit status 22.

## 4. Pluginul + resource pack-ul
Din release-ul GitHub ales; se ia **cel mai recent pack (rNN)** și i se calculează SHA-1 real — configul se generează din el.

In [ ]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
cd "$ROOT/mc"
mkdir -p plugins
REPO="ExMarius/MrCrayfishVehicleMod"
BASE="https://github.com/$REPO/releases/download/$RELEASE_TAG"
A=$(curl -fsSL "https://api.github.com/repos/$REPO/releases/tags/$RELEASE_TAG")
JA=$(echo "$A" | python3 -c 'import sys,json; print([a["name"] for a in json.load(sys.stdin)["assets"] if a["name"].endswith(".jar")][0])')
[ -f plugins/MrCrayfishVehiclePlugin.jar ] || curl -fsSL -o plugins/MrCrayfishVehiclePlugin.jar "$BASE/$JA"
echo "$A" | python3 > /tmp/pack.txt -c '
import re, sys, json
c = [n for n in (a["name"] for a in json.load(sys.stdin)["assets"]) if re.search(r"-r\d+\.zip$", n)]
print(max(c, key=lambda n: int(re.search(r"-r(\d+)\.zip$", n).group(1))))'
PACK=$(cat /tmp/pack.txt)
curl -fsSL -o "/tmp/$PACK" "$BASE/$PACK"
SHA=$(sha1sum "/tmp/$PACK" | awk '{print $1}')
{ echo "PACK_NAME=\"$PACK\""; echo "PACK_URL=\"$BASE/$PACK\""; echo "PACK_SHA1=\"$SHA\""; } >> "$ROOT/env.sh"
echo "✅ $JA + $PACK (sha1: ${SHA:0:12}…)"


## 5. Fișierele serverului
`allow-flight=true` (obligatoriu la vehicule), whitelist forțat, distanțe reglate pentru Colab.

In [ ]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
cd "$ROOT/mc"
echo "eula=true" > eula.txt
cat > server.properties <<'EOF'
motd=Plugins Test - MrCrayfish Vehicles
server-port=25565
online-mode=true
white-list=true
enforce-whitelist=true
allow-flight=true
spawn-protection=0
max-players=8
view-distance=6
simulation-distance=4
sync-chunk-writes=false
difficulty=peaceful
spawn-monsters=false
level-type=minecraft:flat
generate-structures=false
EOF
echo "✅ eula.txt + server.properties scrise."


## 6. Restaurare din Google Drive
Aduce înapoi lumea, datele pluginului, whitelist/ops **și parola panoului** (`panel/auth.json`) salvate la sesiunea anterioară. Prima rulare nu are nimic de restaurat.

In [ ]:
import os, shutil

ROOT = "/content/mcserver"
BACKUP = "/content/drive/MyDrive/plugins-test-backup"
ENTRIES = ["mc/world", "mc/world_nether", "mc/world_the_end",
           "mc/plugins/MrCrayfishVehiclePlugin", "mc/whitelist.json",
           "mc/ops.json", "panel/auth.json"]

from google.colab import drive
drive.mount("/content/drive")

restored = []
for entry in ENTRIES:
    src, dst = os.path.join(BACKUP, entry), os.path.join(ROOT, entry)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        restored.append(entry)
print("✅ Restaurat:", ", ".join(restored) if restored else "nimic (prima rulare)")


## 7. Configul pluginului
Generat după restaurare, cu URL + SHA-1 real ai pack-ului.

In [ ]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
cd "$ROOT/mc"
mkdir -p plugins/MrCrayfishVehiclePlugin
cat > plugins/MrCrayfishVehiclePlugin/config.yml <<EOF
# Generat automat de Plugins_Test.ipynb
resource-pack:
  url: "$PACK_URL"
  sha1: "$PACK_SHA1"
  required: true
  prompt: "Acest resource pack este necesar pentru modelele și sunetele vehiculelor."

physics:
  global-speed-limit: 100.0
  fuel-consumption-factor: 1.0

performance:
  activation-distance: 64.0

gas-pump:
  max-hose-distance: 10.0
  fill-seconds: 15.0

persistence:
  autosave-seconds: 300
EOF
echo "✅ config.yml generat pentru $PACK_NAME."


## 8. Tunelul FRP
`frpc.yaml` cu **ambele proxy-uri** (joc + panou web), în formatul probat cu acest `frps`. Tokenul vine din Secrets — nu e scris nicăieri în fișierele repo-ului.

In [ ]:
%%bash
set -euo pipefail
source /content/mcserver/env.sh
cat > "$ROOT/frp/frpc.yaml" <<EOF
serverAddr: "$FRPS_ADDR"
serverPort: $FRPS_PORT
auth:
  token: "$FRPS_TOKEN"
proxies:
  - name: webpanel
    type: tcp
    localIP: "127.0.0.1"
    localPort: 8081
    remotePort: $FRPS_PANEL_PORT
  - name: minecraft
    type: tcp
    localIP: "127.0.0.1"
    localPort: 25565
    remotePort: $FRPS_GAME_PORT
EOF
chmod 600 "$ROOT/frp/frpc.yaml"
echo "✅ frpc.yaml scris (minecraft:$FRPS_GAME_PORT + webpanel:$FRPS_PANEL_PORT)."


## 9. Pornire: FRP → panou → Minecraft
Folosește scripturile repo-ului nemodificate. La final afișează adresele publice și **parola inițială a panoului** (doar la prima rulare; apoi parola e cea salvată în Drive).

In [ ]:
%%bash
set -uo pipefail
source /content/mcserver/env.sh
cd "$ROOT"

bin/start-frp.sh
sleep 4
tail -n 4 logs/frpc.log || true

bin/start-panel.sh
if [ -f panel/initial-password.txt ]; then
  echo "────────────────────────────────────────"
  echo "🔑 Parola inițială a panoului (salveaz-o!):"
  cat panel/initial-password.txt
  echo "────────────────────────────────────────"
else
  echo "🔑 Parola panoului e cea salvată în Drive (auth.json restaurat)."
fi

bin/start-minecraft.sh
echo "→ Aștept pornirea serverului…"
for i in $(seq 1 72); do
  grep -q 'Done (' logs/server-console.log 2>/dev/null && break
  screen -list 2>/dev/null | grep -q '[.]mcv-minecraft' || { echo "❌ Serverul a căzut:"; tail -n 30 logs/server-console.log; exit 1; }
  sleep 5
done
grep -q 'Done (' logs/server-console.log || { echo "❌ Timeout:"; tail -n 30 logs/server-console.log; exit 1; }
grep 'Done (' logs/server-console.log | tail -1

SESSION=$(screen -list | grep -o '[0-9]*[.]mcv-minecraft' | head -1)
for name in $OP_LIST; do
  screen -S "$SESSION" -p 0 -X stuff "whitelist add $name"$(printf '\r')
  screen -S "$SESSION" -p 0 -X stuff "op $name"$(printf '\r')
  echo "→ OP + whitelist: $name"
done
sleep 2
: > "$ROOT/logs/last-address.txt"
{ echo ""; echo "══════════════════════════════════════════════════"
  echo "🟢 Minecraft:  $FRPS_ADDR:$FRPS_GAME_PORT"
  echo "🟢 Panou web:  http://$FRPS_ADDR:$FRPS_PANEL_PORT"
  echo "══════════════════════════════════════════════════"; } | tee "$ROOT/logs/last-address.txt"


## 10. 💬 Consolă
Trimite o comandă (ex. `list`, `say Salut`, `weather clear`). Panoul web are și el câmp de consolă.

In [ ]:
%%bash
#@title 💬 Comandă către server { display-mode: "form" }
CMD = "list" #@param {type:"string"}
source /content/mcserver/env.sh
SESSION=$(screen -list | grep -o '[0-9]*[.]mcv-minecraft' | head -1) || true
if [ -z "${SESSION:-}" ]; then echo "Serverul nu rulează."; exit 1; fi
screen -S "$SESSION" -p 0 -X stuff "$CMD"$(printf '\r')
sleep 2
tail -n 12 "$ROOT/logs/server-console.log"


## 11. 📊 Stare
Scriptul `status.sh` al repo-ului + jurnalul serverului + locația actuală a runtime-ului Colab.

In [ ]:
%%bash
source /content/mcserver/env.sh
cd "$ROOT"
bin/status.sh || true
echo
echo "── Ultimele loguri:"
tail -n 15 logs/server-console.log 2>/dev/null || echo "(gol)"
echo
echo "── Runtime Colab: $(curl -fsS -m 5 ipinfo.io/city 2>/dev/null || echo '?'), $(curl -fsS -m 5 ipinfo.io/country 2>/dev/null || echo '?')"
cat logs/last-address.txt 2>/dev/null || true


## 12. ⏹️ Oprire grațioasă + backup în Drive
Rulează **mereu înainte** să închizi runtime-ul: oprește Minecraft cu `save-all` + `stop`, apoi salvează în `MyDrive/plugins-test-backup/` lumea, datele pluginului și parola panoului.

In [ ]:
%%bash
source /content/mcserver/env.sh
cd "$ROOT"
bin/stop-minecraft.sh || true
bin/stop-panel.sh || true
bin/stop-frp.sh || true


In [ ]:
import os, shutil, subprocess

ROOT = "/content/mcserver"
BACKUP = "/content/drive/MyDrive/plugins-test-backup"
ENTRIES = ["mc/world", "mc/world_nether", "mc/world_the_end",
           "mc/plugins/MrCrayfishVehiclePlugin", "mc/whitelist.json",
           "mc/ops.json", "panel/auth.json"]

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
os.makedirs(BACKUP, exist_ok=True)

saved = []
for entry in ENTRIES:
    src, dst = os.path.join(ROOT, entry), os.path.join(BACKUP, entry)
    if os.path.exists(src):
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        saved.append(entry)
print("✅ Backup:", ", ".join(saved) if saved else "nimic de salvat")
print(subprocess.run(["du", "-sh", BACKUP], capture_output=True, text=True).stdout.strip())


## 13. 🔑 Resetare parolă panou (opțional)
Dacă ai pierdut parola: oprește panoul (celula de mai sus) și rulează celula următoare — la pornire se generează altă parolă inițială.

In [ ]:
%%bash
source /content/mcserver/env.sh
rm -f "$ROOT/panel/auth.json" "$ROOT/panel/initial-password.txt"
echo "✅ Autentificarea a fost resetată. Rulează din nou celula 9 (doar partea de panou pornește din nou; frp și minecraft sunt deja active / idempotente)."


---
## Depanare

| Simptom | Cauză | Remediu |
|---|---|---|
| `FRPS_ADDR gol` | Secrets neconfigurate | Colab → 🔑 Secrets → adaugă `FRPS_ADDR`, `FRPS_TOKEN` (+`FRPS_PORT` dacă nu e 7000) |
| frpc: `authentication failed` | token greșit/rotit | sincronizează `FRPS_TOKEN` cu configul `frps` |
| frpc: `connection refused` | frps oprit sau firewall | verifică pe host că `frps` rulează și porturile sunt deschise |
| Panou 404/`ERR_EMPTY_RESPONSE` de la exterior | proxy-ul `webpanel` nu a pornit | `tail logs/frpc.log`; remotePort 8081 trebuie liber pe frps |
| „Flying is not enabled" | `allow-flight=false` | re-rulează celula 5 |
| Client refuză pack-ul | sha1/url vechi | re-rulează celulele 4 și 7 cu serverul oprit |
| Lag | CPU Colab | scade `view-distance` în celula 5 |
| Runtime pierdut | cap 12h/idle | Run all din nou; lumea revine din Drive |

**Actualizare plugin:** release nou → oprește serverul → schimbă `RELEASE_TAG` → re-rulează celulele 4, 7 și 9.

**Securitate:** nu da adresa publică necunoscuților; parola panoului ține-o privată; dacă tokenul FRP a fost vreodată partajat, rotește-l pe `frps` și actualizează Secretul.